In [1]:
import torch
import numpy as np
import pandas as pd
import matplotlib
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt

# encode = lambda s: [stoi[c] for c in s]
# decode = lambda l:"".join([itos[i] for i in l])
# data_np = np.fromiter((stoi[c] for c in text),dtype = np.int16 ,count = len(text))
# data = torch.from_numpy(data_np)
# del ds, text
# print('GO, Awais.You are the best.', data.shape)

max_iter = 25000
vocab_size = 1516
n_heads = 8
n_embed = 512
block_size = 256
batch_size = 64
n_layers = 6
eval_interval = 1000
eval_iter = 30
droupout = 0.0
device = 'cuda' if torch.cuda.is_available() else 'cpu'
head_size = n_embed // n_heads
learning_rate = 3e-4

# n = int(0.9 * v)
# data_train = data[:n]
# data_test = data[n:]

# def get_batch(batch):
#   data = data_train if batch == 'train' else data_test
#   ix = torch.randint(len(data) - block_size , (batch_size,))
#   x = torch.stack([data[i:i+block_size] for i in ix]).long()
#   y = torch.stack([data[i+1:i+block_size+1] for i in ix]).long()
#   x, y = x.to(device), y.to(device)
#   return x,y

class head(nn.Module):
  def __init__(self,head_size):
    super().__init__()
    self.key = nn.Linear(n_embed, head_size, bias = False)
    self.query = nn.Linear(n_embed, head_size, bias = False)
    self.value = nn.Linear(n_embed, head_size, bias = False)
    self.register_buffer('tril',torch.tril(torch.ones(block_size,block_size)))
  def forward(self,x):
    B,T,C = x.shape
    k = self.key(x)
    q = self.query(x)
    V = self.value(x)
    # incorporate relationship not situationship(do it in fine tuning)
    wei = q @ k.transpose(-2,-1)*k.shape[-1]**(-0.5)
    wei = wei.masked_fill(self.tril[:T,:T] == 0, float('-inf'))
    wei = F.softmax(wei,dim = -1)
    out = wei @ V
    return out

class MultiHeadedAttention(nn.Module):
  def __init__(self,n_heads, head_size):
    super().__init__()
    self.heads = nn.ModuleList([head(head_size) for _ in range(n_heads)])
    self.proj = nn.Linear(n_heads * head_size , n_embed)

  def forward(self,x):
    eva = torch.cat([h(x) for h in self.heads],dim = -1)
    ans = self.proj(eva)
    return ans

class feed_forward(nn.Module):
  def __init__(self, n_embed):
    super().__init__()
    self.linda = nn.Sequential(
        nn.Linear(n_heads * head_size , n_embed * 4),
        nn.ReLU(),
        nn.Linear(n_embed * 4 , n_embed)
    )
  def forward(self, x):
    out = self.linda(x)
    return out

class block(nn.Module):
  def __init__(self, n_embed , n_heads):
    super().__init__()
    head_size = n_embed // n_heads
    self.cbc = MultiHeadedAttention(n_heads , head_size)
    self.fw = feed_forward( n_embed)
    self.ln1 = nn.LayerNorm(n_embed)
    self.ln2 = nn.LayerNorm(n_embed)


  def forward(self, x):
    x = x + self.cbc(self.ln1(x))
    x = x + self.fw(self.ln2(x))
    return x

class Nosheen_Hina(nn.Module):
  def __init__(self,):
    super().__init__()
    self.token_embedding_table = nn.Embedding(vocab_size,n_embed)
    self.position_embedding_table = nn.Embedding(block_size,n_embed)
    self.blocks = nn.Sequential(*[block(n_embed , n_heads = n_heads) for _ in range(n_layers)])
    self.ln_l = nn.LayerNorm(n_embed)
    self.lm_head = nn.Linear(n_embed , vocab_size)
    self.apply(self._init_weights)

  def _init_weights(self, module):
    if isinstance(module , nn.Linear):
      torch.nn.init.normal_(module.weight, mean = 0.00 , std = 0.02 )
      if module.bias is not None:
        torch.nn.init.zeros_(module.bias)
    elif isinstance(module , nn.Embedding):
      torch.nn.init.normal_(module.weight, mean = 0.00 , std = 0.02 )

  def forward(self, idx , target = None):
    B,T = idx.shape
    tokens = self.token_embedding_table(idx)
    posi = self.position_embedding_table(torch.arange(T,device = device))
    x = tokens + posi
    x = self.blocks(x)
    x = self.ln_l(x)
    logits = self.lm_head(x)
    if target is None:
      loss = None
    else:
      B,T,C = logits.shape
      logits = logits.view(B*T , C)
      target = target.view(B*T)
      loss = F.cross_entropy(logits,target)
    return logits , loss

  def Generator(self , idx , max_token):
    for i in range(max_token):
      logits,loss = self(idx[ : ,-block_size: ])
      B,T,C = logits.shape
      logits = logits[:,-1,:]
      prob = F.softmax(logits , dim = -1)
      ndx = torch.multinomial(prob , num_samples = 1)
      idx = torch.cat((idx, ndx), dim = 1)
    return idx

# model = Nosheen_Hina()
# m = model.to(device)
# print(sum(p.numel() for p in m.parameters())/1e6, 'M parameters')

# @torch.no_grad()
# def evaluate_loss():
#   out = {}
#   model.eval()
#   for split in ['train', 'test']:
#     losses = torch.zeros(eval_iter)
#     for i in range(eval_iter):
#       x_a, y_a = get_batch(split)
#       logits , loss = model(x_a, y_a)
#       losses[i] = loss.item()
#     out[split] = losses.mean()
#   model.train()
#   return out

# optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)
# # Smoothly decays learning rate from 3e-4 down to 3e-5 over 25,000 steps
# scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=max_iter, eta_min=3e-5)

# best_val_loss = float('inf')

# for i in range(max_iter):
#     if (i % eval_interval == 0) or (i == max_iter - 1):
#         losses = evaluate_loss()
#         print(f"step {i}: train loss {losses['train']:.4f}, val loss {losses['test']:.4f}", flush=True)

#         if losses['test'] < best_val_loss:
#             best_val_loss = losses['test']
#             torch.save({
#                 'step': i,
#                 'model_state_dict': model.state_dict(),
#                 'optimizer_state_dict': optimizer.state_dict(),
#                 'val_loss': best_val_loss,
#                 'stoi': stoi,
#                 'itos': itos,
#             }, 'nosheen_hina_best.pt')
#             print(f"--> Saved new best model at step {i} (val_loss: {best_val_loss:.4f})", flush=True)

#     x, y = get_batch('train')
#     logits, loss = model(x, y)
#     optimizer.zero_grad(set_to_none=True)
#     loss.backward()
#     optimizer.step()
#     scheduler.step()  # updates learning rate smoothly each step

# # Save final weights
# torch.save({
#     'step': max_iter,
#     'model_state_dict': model.state_dict(),
#     'stoi': stoi,
#     'itos': itos,
# }, 'nosheen_hina_final.pt')
# print("Training finished and final weights saved successfully!", flush=True)

# Generate and save a 1000-character sample
# context = torch.zeros((1, 1), dtype=torch.long, device=device)
# sample_text = decode(model.Generator(context, max_token=1000)[0].tolist())
# print("\n--- GENERATED SAMPLE ---\n", sample_text)

# with open("generated_sample.txt", "w", encoding="utf-8") as f:
#     f.write(sample_text)


# 1. Locate the attached .pt file automatically
checkpoint_path = 'nosheen_hina_best.pt'
checkpoint = torch.load(checkpoint_path, map_location=device)

# 2. Extract mappings from checkpoint
stoi = checkpoint['stoi']
itos = checkpoint['itos']
vocab_size = 1516  # Force the correct vocabulary size

encode = lambda s: [stoi[c] for c in s if c in stoi]
decode = lambda l: "".join([itos[i] for i in l])

# 3. Instantiate architecture and load trained weights
model = Nosheen_Hina().to(device)
model.load_state_dict(checkpoint['model_state_dict'])
model.eval()

print(f"Teacher Shneela is awake! (Val Loss: {checkpoint.get('val_loss', 'N/A'):.4f})\n")

# 4. Interactive generation loop


Teacher Shneela is awake! (Val Loss: 0.8314)



In [ ]:
while True:
    print('Me Hoon Teacher Shneela!   Kya Poochna he?')
    user_prompt = input("Enter prompt (or 'quit' to exit): ")
    if user_prompt.strip().lower() == 'quit':
        break
    if not user_prompt:
        user_prompt = "The "

    context = torch.tensor([encode(user_prompt)], dtype=torch.long, device=device)
    with torch.no_grad():
        output = model.Generator(context, max_token=350)

    print("\n--- OUTPUT ---")
    print(decode(output[0].tolist()))
    print("-" * 40 + "\n")

Me Hoon Teacher Shneela!   Kya Poochna he?


Enter prompt (or 'quit' to exit):  the artificial intelligence



--- OUTPUT ---
the artificial intelligence between theoretical developments to incorporating arts and crafts mechanisms into their versatility.
2. Homoboma composer Rough emissions Chronographer Coffrey Contest: Observe symmetry behind if there isn't any accessory effects over absorption. Rangers possess elegant metals and sea browners' hours filled with slightly offense, complex, and appl
----------------------------------------

Me Hoon Teacher Shneela!   Kya Poochna he?
